# Example 1 &ndash; Simple 2D Simulation

In example demonstrates setting up a simple 2D simulation in both time- and frequency-domains. FrequenSolve is built on a fast frequency-domain solver; it does not yet have any native time-domain propagators but time-domain simulation is supported by simulating many (often hundreds) of discrete frequencies, then applying the inverse Fourier transform. The frequency-domain solver can be remarkably fast and memory-efficient; so fast that time-domain simulation (with hundreds of frequencies) can be competitive, especially for challenging problems with high-contrast, attenuation, topography, and other challenging effects. However, FrequenSolve's approach is unique and effectively leveraging it requires some insight into how it works. That insight will be built up over the course of these examples.

We begin by importing FrequenSolve modules and creating a new project. FrequenSolve modules can be imported individually or, for convenience, a flat namespace is provided by importing `frequensolve_flat`

## Creating a Project

Projects are the highest level container in FrequenSolve; `Project.path` specifies the working directory. A project configuration file will be created under `[Project.path]/[Project.name].json`; this is used to track (and migrate) FrequenSolve versions and can be used to load the project. An optional 'pretty' name can also be specified to be used for autogenerating reports, etc.. 

In [5]:
import os
import numpy as np
from frequensolve_flat import *

project_path = "./scratch/ex01_simple/"
project = Project(name           = "ex01_simple",
                  pretty_name    = "Simple Simulation (Frequency Domain)",   
                  path           = project_path,
                  load_if_exists = False)

## Defining a Model

Models can be defined a few different ways in FrequenSolve. In this example we'll use the `LayeredModel` class; this builds a layered model from a collection of non-intersecting simple surfaces sandwiching material 'layers' that specify the velocity model. In 2D simulations, a `LayeredModel` is initialized by specifying the x-limits of the model; the z-limits will be implied by the model surfaces. Surfaces must be defined at the top and bottom of the model, as well as between layers. 

For now, we'll build a simple model composed two homogeneous layers separated by a simple flat interface. Supported material properties include:
- __*Vp*__: Compressive velocity
- __*Qp*__: Compressive quality factor
- __*Vs*__: Shear velocity
- __*Qs*__: Shear quality factor
- __*Rho*__: Density

FrequenSolve supports general anisotropy and various parameterizations, API support for these will be added in the near future. For now, we will assume velocity is specified in *km/s* and density in *g/cm^3*.

In [6]:
# New 2D layered model over interval x in [0,1]
model = LayeredModel(dimension = 2, x_limits = [0.0, 1.0])

model.add_surface(z = 0.0)          # Top surface
model.add_layer(                    # Layer 1
   name = "simple",
   properties = { "Vp" : 1.0,
                  "Rho": 1.0}
)
model.add_surface(z = 0.25)         # Interface
model.add_layer(                    # Layer 2 
   name = "simple",
   properties = { "Vp" : 2.0,
                  "Vs" : 1.0,
                  "Rho": 1.0}
)
model.add_surface(z = 1.0)          # Bottom surface

## Meshing

As a finite element method, FrequenSolve operates on a computational mesh. FrequenSolve uses a two-step meshing process:

1. An initial mesh is defined via built-in `MeshGenerators`.
2. Advanced adaptive mesh refinement is used to optimize the mesh for each simulation and frequency. 

Adaptive mesh refinement produces a heirarchy of meshes that are used to solve the frequency-domain problem efficiently. This is an important point, the goal of initial mesh generation is *not* to produce accurate meshes that can accurately resolve waves *nor* is it to accurately capture curvilinear geometries (these are the responsibility of the subsequent adaptive mesh refinement); __*the goal of initial mesh generation is to provide a coarse space for the solver*__. A good target is often ~8 &ndash; 32 elements in each direction.

The `HexMeshGenerator` is a simple mesh generator for `LayeredModel`s; it produces a quad (2D) or hex (3D) mesh that conforms to layers. External tools can also be used to generate an initial mesh, but many tools discard information about surfaces, etc.; mesh refinements will thus be unable to adapt to curvilinear features.

In [7]:
# Define mesh generator
mesh = HexMeshGenerator(n = [16, 16], model = model)

## Boundary Conditions

Boundary Conditions are managed by the `BoundaryConditionManager` class. Boundaries can either be identified via geometry (`label_type = "geometric"`) or defined in the mesh (`label_type = "labeled"`). The API currently supports the following boundary condition types (additional boundary conditions can be added as required):

- `'dirichlet'`:              Fixed pressure/displacement.
- `'neumann'` (free-surface): Fixed normal velocity/traction.
- `'symmetric'`:              Simulates plane symmetry accross the boundary.
- `'pml'`:                    Perfecly matched layer, simulating an absorbing boundary

In [8]:
# New boundery condition manager
BCs = BoundaryConditionManager(label_type = "geometric")

# Free surface BC
BCs += BoundaryCondition(name       = "free_surface",
                         kind       = "neumann",
                         boundaries = ["z_min"])

# PML boundary condition
BCs += BoundaryCondition(name = "pml",
                         kind = "pml",
                         boundaries = ["x_min", "x_max", "z_max"],
                         pml_wavelengths = 2.0,
                         pml_exponent    = 3.0,
                         pml_constant    = 20.0)

## Defining an Acquistion

The `Acquisition` class stores information about source and receiver locations, type, etc.. Currently, all sources in an acquisition will be batched and solved in a single group. We also assume that receiver positions do not vary over sources. In the future we intend to define additional classes to manage entire surveys with many sources and receivers varying by source; we will then extract and solve groups of nearby sources and allow receiver position to be source dependent. For now, memory constraints often require limiting the number of sources in a group to <100. If more sources are required, additional `Simulation`s can be used to separate the sources into more managable groups.

### Sources
FrequenSolve supports `'scalar'` and `'vector'` sources in acoustic domains, and `'vector'` and `'moment'` sources in elastic domains:

- For `'scalar'` sources, `'direction'` is a scalar array scales the amplitude. 
   - 2D & 3D: `[ A ]`
- For `'vector'` sources, `'direction'` is an array with the same dimension as the simulation.
   - 2D: `[ f_x, f_z ]`
   - 3D: `[ f_x, f_y, f_z ]`
- Finally, for `'moment'` sources, `'direction'` is a rank-2 tensor given in Voigt notation.
   - 2D: `[ M_xx, M_yy, M_xy ]`
   - 3D: `[ M_xx, M_yy, M_zz, M_yz, M_xz, M_xy ]`

### Receivers
FrequenSolve supports multicomponent recievers and various receiver types designed to facilitate integration of measurments from diverse site instrumentation. First we define reciever device; devices are derived from the `ReceiverDevice` class, supported devices include:

- `RecieverNode`: A single-node sensor, recording point measurements.
- `RecieverNodeArray`: A single-channel array of nodes, recording average point measurements over the array.
- `RecieverFiber`: DAS fiber (straight or helical), integrating tangential responses over a 1-D length.




In [10]:
# ----------------------------------------------------------------------
# Define Acquisition source, receiver geometry
# ----------------------------------------------------------------------
acq = Acquisition()

# --- Sources ---
coords = []
for x in np.linspace(0.4, 0.6, 3):
   coords.append([x, 0.0])

acq.add_source_group(kind        = "vector",
                     coordinates = coords,
                     direction   = [0.0, 1.0])

# -- Receivers ---
# Define a multi-component device, recording pressure, u_x, and u_z
device = ReceiverNode(name = "geophone")
device.add_component("p"  ,"pressure")
device.add_component("u_z","displacement",[0.0, 1.0])
device.add_component("u_x","displacement",[1.0, 0.0])

# Define reciever coordinates
coords = []
for i, x in enumerate(np.linspace(0.0, 4.0, 1001)):
   coords.append([x, 0.0])

acq.add_receiver_group(name        = "surface_geophones",
                       device      = device,
                       coordinates = coords,
                       frame       = "reference")


TypeError: Acquisition.__init__() missing 1 required positional argument: 'samples'

## Outputs



## Configuring a Simulation

Each project can contain multiple simulations. The `Simulation` class stores general simulation parameters `dimension`, `physics`, etc., that should be provided at initialization. Simulation components such as the model, mesh, boundary conditions, etc. will then be defined and added to the simulation.

We'll start with an acoustic 2D frequency-domain simulation. 

In [2]:
sim = project.new_FD_simulator( name      = "ex01_freq",
                                physics   = "acoustic",
                                dimension = 2,
                                f_list    = [10.0] )

In [3]:
sim = project.new_TD_simulator( name      = "ex01_time",
                                physics   = "acoustic",
                                dimension = 2,
                                f_min     = 1.0,
                                f_max     = 40.0,
                                df        = 1.0 )

## Putting it Together

In [4]:
sim.model = model
sim.mesh  = mesh
sim.BCs = BCs
sim.acquisition = acq




print(sim.to_json(indent=3))